<a href="https://colab.research.google.com/github/aaryachauhan123/AI-ML-Learning/blob/main/Image_classification_using_satelighit_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================
# 1. Upload ZIP file in Colab
# =========================================
from google.colab import files
uploaded = files.upload()



In [ ]:
# =========================================
# 2. Unzip dataset n
# =========================================
import zipfile
import os


zip_filename = list(upload.keys())[0]


with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall("/content/")


dataset_path = "/content/Satellite Image data"   # change if needed
print("Folders inside dataset:", os.listdir(dataset_path))





In [ ]:


# =========================================
# 3. Collect all image paths and labels
# =========================================
image_paths = []
labels = []


class_names = sorted(os.listdir(dataset_path))
print("Class names:", class_names)


for label, class_name in enumerate(class_names):
    class_folder = os.path.join(dataset_path, class_name)
    for file_name in os.listdir(class_folder):
        img_path = os.path.join(class_folder, file_name)
        image_paths.append(img_path)
        labels.append(label)


print("Total images:", len(image_paths))





In [ ]:
# 4. Train-test split
# =========================================
from sklearn.model_selection import train_test_split
import numpy as np


image_paths = np.array(image_paths)
labels = np.array(labels)


train_paths, test_paths, train_labels, test_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)


print("Training samples:", len(train_paths))
print("Testing samples:", len(test_paths))


#class wise sample split
print("Train class counts:", np.bincount(train_labels))
print("Test class counts:", np.bincount(test_labels))



In [ ]:


# Create TensorFlow datasets
# =========================================
import tensorflow as tf


IMG_SIZE = (224, 224)
BATCH_SIZE = 32


def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label


train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))


train_ds = train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)



In [ ]:
# =========================================
# 6. Preprocess for ResNet50
# =========================================
from tensorflow.keras.applications.resnet50 import preprocess_input


train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y),
                        num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (preprocess_input(x), y),
                      num_parallel_calls=tf.data.AUTOTUNE)

# Ensure the dataset is unbatched before applying the final batching
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# 7. Build ResNet50 Transfer Learning Model
# =========================================
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model


num_classes = len(class_names)


base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


base_model.trainable = False


inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation="softmax")(x)


model = Model(inputs, outputs)


In [ ]:
# =========================================
# 8. Compile model
# =========================================
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


model.summary()


In [ ]:
# =========================================
# 9. Train model
# =========================================
history = model.fit(
    train_ds,
    epochs=5
)




Epoch 1/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 733s 5s/step - accuracy: 0.9321 - loss: 0.2120
Epoch 2/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 745s 5s/step - accuracy: 0.9953 - loss: 0.0244
Epoch 3/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 721s 5s/step - accuracy: 0.9967 - loss: 0.0156
Epoch 4/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 716s 5s/step - accuracy: 0.9984 - loss: 0.0104
Epoch 5/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 718s 5s/step - accuracy: 0.9971 - loss: 0.0103


In [ ]:
# =========================================
# 10. Evaluate on test data
# =========================================
loss, accuracy = model.evaluate(test_ds)
print("\nTest Accuracy:", accuracy)




In [ ]:


# =========================================
# 11. Predict on test data
# =========================================
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_labels


print("\nFirst 10 true labels:   ", y_true[:10])
print("First 10 predicted labels:", y_pred[:10])


In [ ]:
# =========================================
# 12. Confusion Matrix
# =========================================
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt


cm = confusion_matrix(y_true, y_pred)


plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()


plt.xticks(np.arange(len(class_names)), class_names, rotation=45)
plt.yticks(np.arange(len(class_names)), class_names)


for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="black")


plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 13. Classification Report
# =========================================
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))






import numpy as np


wrong_indices = np.where(y_true != y_pred)[0]
print("Number of wrong predictions:", len(wrong_indices))


print("Wrong indices:", wrong_indices[:10])






import numpy as np


wrong_indices = np.where(y_true != y_pred)[0]
print("Number of wrong predictions:", len(wrong_indices))


print("Wrong indices:", wrong_indices[:10])


